In [1]:
from cobra.io import load_json_model
model = load_json_model('models/Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.concoct_out.9.contigs__.RAST.json')
print(model)

Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.concoct_out.9.contigs__.RAST


In [2]:
print(type(model.solver))

<class 'optlang.glpk_interface.Model'>


In [3]:
from modelseedpy.core.mstemplate import MSTemplateBuilder
from json import load
with open("../../../ModelSEEDTemplates/templates/v6.0/Core-V5.2.json") as fh:
    template_core = MSTemplateBuilder.from_dict(load(fh)).build()


from modelseedpy.core.msatpcorrection import load_default_medias
default_medias = load_default_medias()
print(f'loaded {len(default_medias)} medias')
for media, min_obj in default_medias:
    print(media.id, media.get_media_constraints())
    break


from modelseedpy import MSATPCorrection
atp_correction = MSATPCorrection(model, template_core, default_medias,
                                 compartment='c0', atp_hydrolysis_id='ATPM_c0', 
                                 load_default_medias=False)

modelseedpy 0.4.3
loaded 54 medias
Glc.O2 {'cpd00027_e0': (-1, 1000), 'cpd00007_e0': (-1000, 1000), 'cpd00001_e0': (-1000, 1000), 'cpd00067_e0': (-1000, 1000)}


In [4]:
for rxn in model.reactions:
    if "_c1" in rxn.id:
        print(rxn.id)

In [5]:
media_eval = atp_correction.evaluate_growth_media()
atp_correction.determine_growth_media()
atp_correction.apply_growth_media_gapfilling()
atp_correction.expand_model_to_genome_scale()
tests = atp_correction.build_tests()

No gapfilling solution found before filtering for Etho activating rxn00062_c0
No gapfilling solution found before filtering for mal-L activating rxn00062_c0
No gapfilling solution found before filtering for Pyr.SO4 activating rxn00062_c0
No gapfilling solution found before filtering for H2.SO4 activating rxn00062_c0
No gapfilling solution found before filtering for empty activating rxn00062_c0
No gapfilling solution found before filtering for Light activating rxn00062_c0
No gapfilling solution found before filtering for ANME activating rxn00062_c0
No gapfilling solution found before filtering for Methane activating rxn00062_c0


In [6]:
for m, stats in atp_correction.media_gapfill_stats.items():
    # print(m.id)
    if "Lac" in m.id:
        print(m.id)
        display(stats)

LLac.O2


{'reversed': {},
 'new': {'rxn14426_c0': '>',
  'rxn00499_c0': '>',
  'rxn10122_c0': '>',
  'rxn00225_c0': '<',
  'rxn00173_c0': '>',
  'rxn13689_c0': '>',
  'rxnCYTCBB3_c0': '>'},
 'media': <modelseedpy.core.msmedia.MSMedia at 0x124b99d50>,
 'target': 'rxn00062_c0',
 'minobjective': 2,
 'binary_check': False}

LLac.SO4.H2


{'reversed': {},
 'new': {'rxn00499_c0': '>', 'rxn00225_c0': '<', 'rxn00173_c0': '>'},
 'media': <modelseedpy.core.msmedia.MSMedia at 0x124858220>,
 'target': 'rxn00062_c0',
 'minobjective': 0.01,
 'binary_check': False}

LLac.SO4


{'reversed': {},
 'new': {'rxn00499_c0': '>', 'rxn00225_c0': '<', 'rxn00173_c0': '>'},
 'media': <modelseedpy.core.msmedia.MSMedia at 0x1248586d0>,
 'target': 'rxn00062_c0',
 'minobjective': 0.01,
 'binary_check': False}

In [7]:
tests

[{'media': <modelseedpy.core.msmedia.MSMedia at 0x124858af0>,
  'is_max_threshold': True,
  'threshold': 1e-05,
  'objective': 'rxn00062_c0'},
 {'media': <modelseedpy.core.msmedia.MSMedia at 0x124b9b9d0>,
  'is_max_threshold': True,
  'threshold': 4.000000000000022,
  'objective': 'rxn00062_c0'},
 {'media': <modelseedpy.core.msmedia.MSMedia at 0x124be1300>,
  'is_max_threshold': True,
  'threshold': 2.4000000000000132,
  'objective': 'rxn00062_c0'},
 {'media': <modelseedpy.core.msmedia.MSMedia at 0x124be16c0>,
  'is_max_threshold': True,
  'threshold': 2.4000000000000132,
  'objective': 'rxn00062_c0'}]

In [8]:
new_tests = []
for t in tests:
    if "." not in t['media'].id:
        new_tests.append(t)
    print(t['media'].id,
        t['threshold'], "ATP were produced",
        atp_correction.media_gapfill_stats[t['media']])
    
new_tests

empty 1e-05 ATP were produced None
Glc 4.000000000000022 ATP were produced {'reversed': {}, 'new': {}}
Glc.DMSO 2.4000000000000132 ATP were produced {'reversed': {}, 'new': {}}
Glc.TMAO 2.4000000000000132 ATP were produced {'reversed': {}, 'new': {}}


[{'media': <modelseedpy.core.msmedia.MSMedia at 0x124858af0>,
  'is_max_threshold': True,
  'threshold': 1e-05,
  'objective': 'rxn00062_c0'},
 {'media': <modelseedpy.core.msmedia.MSMedia at 0x124b9b9d0>,
  'is_max_threshold': True,
  'threshold': 4.000000000000022,
  'objective': 'rxn00062_c0'}]

In [9]:
with open("../../../ModelSEEDTemplates/templates/v6.0/GramNegModelTemplateV6.json") as fh:
    template_gramneg = MSTemplateBuilder.from_dict(load(fh)).build()

from modelseedpy import MSGapfill
gapfill = MSGapfill(model, default_gapfill_templates=[template_gramneg], test_conditions=new_tests, default_target='bio1')

from modelseedpy import MSMedia
media = MSMedia.from_dict({'cpd00067': 1000.0,
 'cpd00058': 1000.0,
 'cpd00013': 1000.0,
 'cpd00244': 1000.0,
 'cpd00205': 1000.0,
 'cpd00034': 1000.0,
 'cpd11574': 1000.0,
 'cpd00971': 1000.0,
 'cpd00048': 1000.0,
 'cpd00030': 1000.0,
 'cpd00305': 100.0,
 'cpd00001': 1000.0,
 'cpd10516': 1000.0,
 'cpd00007': 1000.0,
 'cpd00159': 100.0,
 'cpd25960': 1000.0,
 'cpd00027': 10.0,
 "cpd00009": 100,
 'cpd00063': 1000.0,
 'cpd00149': 1000.0,
 'cpd00254': 1000.0,
 'cpd00099': 1000.0})

media_glucose = MSMedia.from_dict({
    'cpd00149': (-100, 100),
    'cpd00099': (-100, 100),
    'cpd00067': (-100, 100),
    'cpd00063': (-100, 100),
 'cpd00058': (-100, 100),
 'cpd00048': (-100, 100),
 'cpd00034': (-100, 100),
 'cpd00030': (-100, 100),
 'cpd00013': (-100, 100),
 'cpd00009': (-100, 100),
 'cpd00001': (-100, 100),
 'cpd00007': (-10, 100),
 'cpd00205': (-100, 100),
 'cpd00254': (-100, 100),
 'cpd00971': (-100, 100),
 'cpd10515': (-100, 100),
 'cpd10516': (-100, 100),
 'cpd11574': (-100, 100),
 'cpd00244': (-100, 100),
 'cpd00027': (-5, 100)})

gapfill_res = gapfill.run_gapfilling(media)

In [10]:
gapfill_res

{'reversed': {},
 'new': {'EX_cpd00048_e0': '<',
  'EX_cpd00063_e0': '<',
  'EX_cpd00099_e0': '<',
  'DM_cpd02701_c0': '>',
  'rxn00359_c0': '<',
  'rxn12008_c0': '<',
  'rxn03086_c0': '<',
  'rxn08333_c0': '>',
  'rxn00738_c0': '<',
  'rxn02898_c0': '>',
  'rxn10336_c0': '>',
  'rxn02504_c0': '>',
  'rxn02988_c0': '<',
  'rxn00470_c0': '>',
  'rxn00128_c0': '<',
  'rxn00152_c0': '>',
  'rxn00526_c0': '>',
  'rxn12512_c0': '>',
  'rxn01739_c0': '>',
  'rxn01974_c0': '>',
  'rxn01640_c0': '>',
  'rxn00068_c0': '<',
  'rxn00060_c0': '>',
  'rxn10197_c0': '>',
  'rxn05293_c0': '>',
  'rxn09202_c0': '>',
  'rxn05514_c0': '<',
  'rxn02212_c0': '>',
  'rxn09427_c0': '>',
  'rxn00692_c0': '>',
  'rxn02160_c0': '>',
  'rxn01362_c0': '<',
  'rxn00727_c0': '>',
  'rxn03852_c0': '>',
  'rxn00375_c0': '>',
  'rxn10213_c0': '>',
  'rxn05229_c0': '>',
  'rxn01644_c0': '>',
  'rxn05039_c0': '>',
  'rxn09180_c0': '>',
  'rxn00295_c0': '>',
  'rxn00029_c0': '>',
  'rxn10291_c0': '>',
  'rxn02774_c0': '

In [11]:
gapfill.test_gapfill_database(media)

True

In [12]:
# print(type(gapfill.gfmodel))
# print(len(gapfill.gfmodel.reactions))
# gapfill.gfmodel.medium = {'EX_cpd00067_e0': 1000.0,
#  'EX_cpd00058_e0': 1000.0,
#  'EX_cpd00013_e0': 1000.0,
#  'EX_cpd00244_e0': 1000.0,
#  'EX_cpd00205_e0': 1000.0,
#  'EX_cpd00034_e0': 1000.0,
#  'EX_cpd11574_e0': 1000.0,
#  'EX_cpd00971_e0': 1000.0,
#  'EX_cpd00048_e0': 1000.0,
#  'EX_cpd00030_e0': 1000.0,
#  'EX_cpd00305_e0': 100.0,
#  'EX_cpd00001_e0': 1000.0,
#  'EX_cpd10516_e0': 1000.0,
#  'EX_cpd00007_e0': 1000.0,
#  'EX_cpd00159_e0': 100.0,
# #  'EX_cpd25960_e0': 1000.0,
#  "EX_cpd00009_e0": 100,
#  'EX_cpd00063_e0': 1000.0,
#  'EX_cpd00149_e0': 1000.0,
#  'EX_cpd00254_e0': 1000.0,
#  'EX_cpd00099_e0': 1000.0}
# gapfill.gfmodel.reactions.bio1.lower_bound = 0.01
# gapfill.gfmodel.reactions.bio1
# gapfill.gfmodel.optimize()

In [13]:
gapfill.integrate_gapfill_solution(gapfill_res)
print(f"Reactions in model after integration: {len(model.reactions)}")

Reactions in model after integration: 970


In [14]:
import math
def integrate_to_model_medium(media, model, prefix='EX_'):
    medium = {}
    for cpd, (lb, ub) in media.get_media_constraints().items():
        rxn_exchange = f'{prefix}{cpd}'
        if rxn_exchange in model.reactions:
            medium[rxn_exchange] = math.fabs(lb)
        else:
            print('not in model', cpd)
    return medium

# integrate_gapfill_solution modifies model in-place, so use model directly
model.medium = integrate_to_model_medium(media, model)
model.objective = 'bio1'
sol = model.optimize()
print(f"Biomass flux: {sol.objective_value}")
model.summary()

not in model cpd25960_e0
Biomass flux: 0.2351386567209841


Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00007_e0,EX_cpd00007_e0,0.01077,0,0.00%
cpd00009_e0,EX_cpd00009_e0,0.274,0,0.00%
cpd00013_e0,EX_cpd00013_e0,1.712,0,0.00%
cpd00027_e0,EX_cpd00027_e0,10,6,99.96%
cpd00030_e0,EX_cpd00030_e0,0.001795,0,0.00%
cpd00034_e0,EX_cpd00034_e0,0.001795,0,0.00%
cpd00048_e0,EX_cpd00048_e0,0.05202,0,0.00%
cpd00058_e0,EX_cpd00058_e0,0.001795,0,0.00%
cpd00063_e0,EX_cpd00063_e0,0.001795,0,0.00%
cpd00099_e0,EX_cpd00099_e0,0.001795,0,0.00%


In [ ]:
model.objective.expression

1.0*bio1 - 1.0*bio1_reverse_b18f7

: 